[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/erickcmendes/CEUB/blob/main/Otimiza%C3%A7%C3%A3o%20para%20Ci%C3%AAncia%20de%20Dados/crypto-portfolio-optimization/notebooks/01_solver_check_colab.ipynb)

# Verificação de Solvers no Google Colab (Task 01-01 / T-002)

**Disciplina:** Otimização Aplicada à Ciência de Dados - CEUB  
**Objetivo:** Instalar e validar os solvers exatos exigidos pelo curso:  
- **CBC** (Programação Linear / Inteira - MILP)  
- **Ipopt** (Programação Não Linear Contínua - NLP)  
- **Couenne** (Programação Não Linear Inteira Mista - MINLP)  

> **Rastreabilidade:** `FR-02`, `NFR-02`, `D-003`, `T-002`.

## 1. Instalação dos Solvers e Dependências

Executamos a instalação via extensões binárias oficiais do IDAES (compiladas para Linux x86_64, padrão do Google Colab) e via `apt-get` como redundância.

In [ ]:
# FR-02, NFR-02: Instalação automatizada dos solvers no ambiente Colab
import sys
import os
import time
import subprocess
from pathlib import Path

start_time = time.time()
print(f"Python Version: {sys.version}")

print("Instalando Pyomo e idaes-pse...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyomo", "idaes-pse"], check=True)

print("Obtendo binários pré-compilados do IDAES (cbc, ipopt, couenne, bonmin)...")
subprocess.run(["idaes", "get-extensions"], check=True)

idaes_bin = Path.home() / ".idaes" / "bin"
if idaes_bin.exists() and str(idaes_bin) not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{idaes_bin}:{os.environ.get('PATH', '')}"

elapsed = time.time() - start_time
print(f"Instalação concluída com sucesso em {elapsed:.2f} segundos.")


## 2. Teste Toy 1: CBC (Modelo Linear)

Problema:
$$\max 3x_1 + 2x_2$$
sujeito a:
$$x_1 + x_2 \le 4$$
$$x_1 \le 2, \quad x_1, x_2 \ge 0$$

In [ ]:
# FR-02, T-002: Teste CBC
import pyomo.environ as pyo

m_cbc = pyo.ConcreteModel(name="Toy_CBC")
m_cbc.x1 = pyo.Var(domain=pyo.NonNegativeReals, bounds=(0, 2))
m_cbc.x2 = pyo.Var(domain=pyo.NonNegativeReals)
m_cbc.c1 = pyo.Constraint(expr=m_cbc.x1 + m_cbc.x2 <= 4)
m_cbc.obj = pyo.Objective(expr=3 * m_cbc.x1 + 2 * m_cbc.x2, sense=pyo.maximize)

opt_cbc = pyo.SolverFactory("cbc")
res_cbc = opt_cbc.solve(m_cbc)

print("Status CBC:", res_cbc.solver.termination_condition)
print(f"Solução: x1 = {pyo.value(m_cbc.x1)}, x2 = {pyo.value(m_cbc.x2)}, Obj = {pyo.value(m_cbc.obj)}")
assert abs(pyo.value(m_cbc.obj) - 10.0) < 1e-4, "Erro no teste CBC"

## 3. Teste Toy 2: Ipopt (Modelo Não-Linear Contínuo)

Problema:
$$\min (x_1 - 1)^2 + (x_2 - 2)^2$$
sujeito a:
$$x_1 + x_2 = 3$$
$$x_1, x_2 \ge 0$$

In [ ]:
# FR-02, T-002: Teste Ipopt
m_ipopt = pyo.ConcreteModel(name="Toy_Ipopt")
m_ipopt.x1 = pyo.Var(domain=pyo.NonNegativeReals, initialize=1.0)
m_ipopt.x2 = pyo.Var(domain=pyo.NonNegativeReals, initialize=1.0)
m_ipopt.c1 = pyo.Constraint(expr=m_ipopt.x1 + m_ipopt.x2 == 3)
m_ipopt.obj = pyo.Objective(expr=(m_ipopt.x1 - 1)**2 + (m_ipopt.x2 - 2)**2, sense=pyo.minimize)

opt_ipopt = pyo.SolverFactory("ipopt")
res_ipopt = opt_ipopt.solve(m_ipopt)

print("Status Ipopt:", res_ipopt.solver.termination_condition)
print(f"Solução: x1 = {pyo.value(m_ipopt.x1):.4f}, x2 = {pyo.value(m_ipopt.x2):.4f}, Obj = {pyo.value(m_ipopt.obj):.4f}")
assert abs(pyo.value(m_ipopt.obj) - 0.0) < 1e-4, "Erro no teste Ipopt"

## 4. Teste Toy 3: Couenne (Modelo Não-Linear Inteiro Misto - MINLP)

Problema:
$$\min x_1^2 + x_2^2 - 4 x_1 y$$
sujeito a:
$$x_1 + x_2 \ge 2$$
$$x_1, x_2 \ge 0, \quad y \in \{0, 1\}$$
$$x_1 \le 3 y$$

In [ ]:
# FR-02, T-002: Teste Couenne
m_couenne = pyo.ConcreteModel(name="Toy_Couenne")
m_couenne.x1 = pyo.Var(domain=pyo.NonNegativeReals, bounds=(0, 3), initialize=1.0)
m_couenne.x2 = pyo.Var(domain=pyo.NonNegativeReals, bounds=(0, 3), initialize=1.0)
m_couenne.y = pyo.Var(domain=pyo.Binary, initialize=1)
m_couenne.c1 = pyo.Constraint(expr=m_couenne.x1 + m_couenne.x2 >= 2)
m_couenne.c2 = pyo.Constraint(expr=m_couenne.x1 <= 3 * m_couenne.y)
m_couenne.obj = pyo.Objective(expr=m_couenne.x1**2 + m_couenne.x2**2 - 4 * m_couenne.x1 * m_couenne.y, sense=pyo.minimize)

opt_couenne = pyo.SolverFactory("couenne")
res_couenne = opt_couenne.solve(m_couenne)

print("Status Couenne:", res_couenne.solver.termination_condition)
print(f"Solução: x1 = {pyo.value(m_couenne.x1):.4f}, x2 = {pyo.value(m_couenne.x2):.4f}, y = {pyo.value(m_couenne.y)}, Obj = {pyo.value(m_couenne.obj):.4f}")
print("Todos os 3 solvers validados com sucesso!")